# 08 — Quantization Analysis

Analyze the impact of post-training quantization (PTQ) on accuracy and model size.
This notebook converts trained Keras models to TFLite (float32, dynamic range int8, full int8),
evaluates accuracy on the test set, and reports size and speed characteristics.


In [ ]:
import os, json, yaml, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, accuracy_score, f1_score
sns.set_context('talk'); sns.set_style('whitegrid')


Paths, config and data

In [ ]:
def detect_root():
    cwd = Path.cwd()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if all((p/d).exists() for d in ['config','data','scripts','notebooks']):
            return p
    return cwd

ROOT = detect_root()
CONFIG_PATH = ROOT/'config'/'config.yaml'
SPLITS_DIR  = ROOT/'data'/'splits'
NEURAL_DIR  = ROOT/'data'/'processed'/'neural'
PLOTS_DIR   = ROOT/'results'/'plots'; PLOTS_DIR.mkdir(parents=True, exist_ok=True)
EVALS_DIR   = ROOT/'results'/'evaluations'; EVALS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = ROOT/'models'/'trained'/'neural'; MODELS_DIR.mkdir(parents=True, exist_ok=True)
TFLITE_DIR  = ROOT/'models'/'tflite'; TFLITE_DIR.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(open(CONFIG_PATH,'r'))
dcfg = cfg.get('config',{})
classes = dcfg.get('classes',[])
num_classes = len(classes)
classes


Load Test set (neural)

In [ ]:
from tensorflow.keras.preprocessing.image import smart_resize

def read_manifest_stems(path: Path):
    stems=set()
    if not path.exists(): return stems
    for line in open(path,'r'):
        line=line.strip()
        if not line: continue
        rel,_ = line.split(',',1)
        stems.add(Path(rel).stem)
    return stems

test_stems = read_manifest_stems(SPLITS_DIR/'test.txt')

def collect_neural_for_split(split_stems, class_names):
    X_list, y_list = [], []
    name_to_idx = {c:i for i,c in enumerate(class_names)}
    for c in class_names:
        cdir = NEURAL_DIR/c
        if not cdir.exists(): continue
        for f in cdir.glob('*.npy'):
            base=f.stem.split('_seg')[0]
            if base in split_stems:
                arr = np.load(f)
                if arr.ndim==3: arr = np.transpose(arr,(1,2,0))
                elif arr.ndim==2: arr = arr[:,:,None]
                else: continue
                X_list.append(arr.astype(np.float32))
                y_list.append(name_to_idx[c])
    if not X_list: return np.empty((0,)), np.empty((0,),dtype=int)
    X = np.stack(X_list, axis=0); y = np.array(y_list, dtype=int)
    return X, y

X_test, y_test = collect_neural_for_split(test_stems, classes)
X_test = np.array([smart_resize(x, (64, 101)) for x in X_test])
X_test = np.mean(X_test, axis=-1, keepdims=True).astype(np.float32)
X_test = (X_test - X_test.min()) / (X_test.max() - X_test.min() + 1e-7)
X_test.shape, len(y_test)


Load trained models to convert

In [ ]:
teacher_path = MODELS_DIR/'teacher_cnn.h5'
student_path = MODELS_DIR/'student_cnn_distilled_nb.h5'
yamnet_path  = MODELS_DIR/'yamnet_ft.h5'
mobilenet_path = MODELS_DIR/'mobilenet_ft.h5'

models_to_convert = []
for p in [teacher_path, student_path, yamnet_path, mobilenet_path]:
    if p.exists():
        models_to_convert.append((p.stem, keras.models.load_model(p)))
        print('Loaded', p)
    else:
        print('[WARN] Missing model:', p)
len(models_to_convert)


TFL conversion helpers

In [ ]:
def convert_tflite(model, repr_ds=None, full_int8=False, dynamic_range=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if full_int8:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if repr_ds is not None:
            def rep_gen():
                for i in range(min(200, len(repr_ds))):
                    yield [repr_ds[i:i+1].astype(np.float32)]
            converter.representative_dataset = rep_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
    elif dynamic_range:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    return tflite_model

def save_tflite(tflite_bytes, out_path: Path):
    out_path.write_bytes(tflite_bytes)
    size_kb = out_path.stat().st_size/1024
    print(f'Saved {out_path.name} — {size_kb:.1f} KB')
    return size_kb

def eval_tflite(tflite_bytes, X, y):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    int8 = (input_details['dtype'] == np.int8)
    preds = []
    for i in range(len(X)):
        x = X[i:i+1]
        if int8:
            scale, zero = input_details['quantization']
            x_q = (x/scale + zero).astype(np.int8)
            interpreter.set_tensor(input_details['index'], x_q)
        else:
            interpreter.set_tensor(input_details['index'], x.astype(np.float32))
        interpreter.invoke()
        y_pred = interpreter.get_tensor(output_details['index'])
        preds.append(y_pred)
    preds = np.vstack(preds)
    y_hat = np.argmax(preds, axis=1)
    acc = accuracy_score(y, y_hat)
    f1  = f1_score(y, y_hat, average='weighted')
    return acc, f1


Convert, evaluate and summarize results

In [ ]:
summary = []
for name, model in models_to_convert:
    print(f'\n=== {name} ===')
    # Float32 baseline
    tflite_f32 = convert_tflite(model)
    size_f32 = save_tflite(tflite_f32, TFLITE_DIR/f'{name}_f32.tflite')
    acc_f32, f1_f32 = eval_tflite(tflite_f32, X_test, y_test)

    # Dynamic range
    tflite_dr = convert_tflite(model, dynamic_range=True)
    size_dr = save_tflite(tflite_dr, TFLITE_DIR/f'{name}_dr.tflite')
    acc_dr, f1_dr = eval_tflite(tflite_dr, X_test, y_test)

    # Full INT8 (with representative dataset)
    tflite_i8 = convert_tflite(model, repr_ds=X_test, full_int8=True)
    size_i8 = save_tflite(tflite_i8, TFLITE_DIR/f'{name}_int8.tflite')
    acc_i8, f1_i8 = eval_tflite(tflite_i8, X_test, y_test)

    summary.append({
        'model': name,
        'size_f32_kb': size_f32, 'acc_f32': acc_f32, 'f1_f32': f1_f32,
        'size_dr_kb': size_dr, 'acc_dr': acc_dr, 'f1_dr': f1_dr,
        'size_int8_kb': size_i8, 'acc_int8': acc_i8, 'f1_int8': f1_i8,
    })

import pandas as pd
df = pd.DataFrame(summary)
display(df)
df.to_csv(EVALS_DIR/'quantization_summary.csv', index=False)

# Plot sizes
plt.figure(figsize=(10,5))
labels = []
sizes = []
for m in summary:
    labels += [f"{m['model']}\nFloat32", f"{m['model']}\nDynRange", f"{m['model']}\nINT8"]
    sizes  += [m['size_f32_kb'], m['size_dr_kb'], m['size_int8_kb']]
plt.bar(labels, sizes, color=['#4c72b0','#55a868','#c44e52']*(len(summary)))
plt.ylabel('KB'); plt.title('Model Sizes after Quantization')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(PLOTS_DIR/'quantization_sizes.png', dpi=150); plt.show()
print('Saved summary to', EVALS_DIR/'quantization_summary.csv')
